# Aula 03 — BoW, N-grams e TF-IDF · acompanhamento

**Origem:** `IAST-ProcessamentodeLinguagemNatural-Fase04_aula03-v1.pptx` · **Kernel:** `.venv`

## O contrato deste notebook

Diferente das aulas 01 e 02. Aqui:

- **Eu escrevi:** as células de código do professor, na ordem, prontas para rodar; o corpus rotulado que faltava; o mapa de dependências e os avisos de "esta célula quebra".
- **Você escreve:** o porquê. Cada bloco termina em **Sua vez**, com perguntas em aberto. Não há resposta pronta em lugar nenhum deste notebook.

Preencha durante a aula, mesmo torto. Depois me peça para revisar e eu corrijo o que estiver errado.

## Duas coisas que você precisa saber antes de começar

**1. Cinco células não rodam como estão no slide.** Os slides 19, 35, 38, 53 e 55 usam variáveis que nenhum slide define (`y`, `reviews_corpus`, `labels`) ou pedem `min_df=5` num corpus de 4 documentos. Consertei o mínimo para rodarem e marquei cada conserto com `# [meu]`.

**2. Os números nos comentários dos slides não batem com o que o código produz** — em praticamente todos. Não conferi para você: onde há divergência, a pergunta em **Sua vez** te manda comparar e explicar. É o exercício.

---
## Corpus rotulado — acréscimo meu, não é do professor

Os slides 38, 53 e 55 assumem `reviews_corpus` e `labels` que nenhum slide define. Montei 64 reviews em PT-BR (32 positivas, 32 negativas) para essas células rodarem.

É só o combustível para essas células rodarem — nenhuma pergunta do notebook é sobre ele.

In [1]:
# [meu] Corpus rotulado usado pelos slides 35, 38, 53 e 55. Não faz parte da aula.
reviews_pos = [
    'Não tive problema, produto bom e entrega rápida',
    'Gostei muito, não demorou para chegar',
    'Não veio quebrado, produto bom, recomendo muito',
    'Recomendo muito, não tive problema com a entrega',
    'Não demorou nada, qualidade boa, gostei muito',
    'Produto bom, não tive problema com o atendimento',
    'Gostei muito do produto, não veio quebrado',
    'Não demorou, entrega rápida, recomendo muito',
    'Não tive problema nenhum, qualidade boa',
    'Recomendo muito o produto, não demorou nada',
    'Não veio quebrado e chegou rápido, gostei muito',
    'Produto bom, gostei muito, não tive problema',
    'Não demorou para chegar, atendimento bom',
    'Gostei muito da qualidade, não veio quebrado',
    'Não tive problema com o vendedor, recomendo muito',
    'Entrega rápida, não demorou, produto bom',
    'Não veio quebrado, qualidade boa, gostei muito',
    'Recomendo muito, não tive problema nenhum',
    'Não demorou e o produto é bom, gostei muito',
    'Qualidade boa, não veio quebrado, recomendo muito',
    'Não tive problema, chegou rápido, produto bom',
    'Gostei muito, não veio quebrado, entrega rápida',
    'Não demorou nada, recomendo muito esse vendedor',
    'Produto bom, não tive problema, qualidade boa',
    'Não veio quebrado, gostei muito do atendimento',
    'Recomendo muito, entrega rápida, não demorou',
    'Não tive problema com a qualidade, produto bom',
    'Gostei muito, chegou rápido, não demorou',
    'Não gostei da embalagem mas recomendo muito o produto',
    'Não gostei do preço mas a qualidade boa compensa',
    'Não gostei da cor mas não tive problema com nada',
    'Não gostei do manual mas o produto bom funciona',
]
reviews_neg = [
    'Não gostei, produto veio quebrado',
    'Não recomendo, entrega demorou muito',
    'Não gostei do produto, tive problema com a entrega',
    'Não recomendo, qualidade ruim, veio quebrado',
    'Não gostei, demorou muito para chegar',
    'Não recomendo esse produto, tive problema',
    'Não gostei da qualidade, veio quebrado',
    'Não recomendo, produto ruim, demorou muito',
    'Não gostei do atendimento, tive problema',
    'Não recomendo, veio quebrado e demorou',
    'Não gostei, tive problema com o vendedor',
    'Não recomendo a loja, qualidade ruim',
    'Não gostei do produto bom que veio quebrado',
    'Não recomendo, demorou muito e veio quebrado',
    'Não gostei, qualidade ruim, tive problema',
    'Não recomendo, produto veio quebrado',
    'Não gostei da entrega, demorou muito',
    'Não recomendo, tive problema com a qualidade',
    'Não gostei, veio quebrado, atendimento ruim',
    'Não recomendo, demorou muito para chegar',
    'Não gostei do vendedor, produto veio quebrado',
    'Não recomendo, tive problema com o produto',
    'Não gostei, demorou e veio quebrado',
    'Não recomendo, qualidade ruim e demorou muito',
    'Não gostei da qualidade, tive problema',
    'Não recomendo, veio quebrado, entrega ruim',
    'Não gostei, produto ruim, demorou muito',
    'Não recomendo, tive problema com a entrega',
    'Não tive problema com a entrega mas o produto veio quebrado',
    'Não tive problema com o vendedor mas a qualidade ruim decepciona',
    'Recomendo muito a loja mas não gostei do produto que veio quebrado',
    'Não demorou muito mas veio quebrado, não recomendo',
]
reviews_corpus = reviews_pos + reviews_neg
labels = [1] * len(reviews_pos) + [0] * len(reviews_neg)
print(f'{len(reviews_corpus)} documentos | {labels.count(1)} positivos | {labels.count(0)} negativos')

64 documentos | 32 positivos | 32 negativos


---
# Bloco 1 · Bag of Words (slides 15–19)

## Slide 15 — BoW manualmente, sem libs

Antes de rodar, escreva o que você espera: qual vai ser o tamanho do vocabulário, e o que a matriz vai ter em cada linha e cada coluna.

In [2]:
# BoW manual — 30 linhas para entender o que sklearn faz
from collections import Counter
corpus = [
    'Adorei o produto, entrega rápida',
    'Produto chegou quebrado',
    'Entrega rápida, atendimento bom',
]
# 1. Tokenizar (simplificado)
docs_tokens = [doc.lower().replace(',','').split() for doc in corpus]
# 2. Construir vocabulário (palavras únicas ordenadas)
vocab = sorted(set(t for doc in docs_tokens for t in doc))
print(f'Vocabulário ({len(vocab)} palavras):', vocab)
# 3. Construir matriz documento-termo
matriz = []
for tokens in docs_tokens:
    contagem = Counter(tokens)
    linha = [contagem.get(p, 0) for p in vocab]
    matriz.append(linha)
import pandas as pd
df = pd.DataFrame(matriz, columns=vocab)
print(df)

Vocabulário (9 palavras): ['adorei', 'atendimento', 'bom', 'chegou', 'entrega', 'o', 'produto', 'quebrado', 'rápida']
   adorei  atendimento  bom  chegou  entrega  o  produto  quebrado  rápida
0       1            0    0       0        1  1        1         0       1
1       0            0    0       1        0  0        1         1       0
2       0            1    1       0        1  0        0         0       1


### Sua vez

1. A tokenização aqui é `.lower().replace(',','').split()`. Depois de tudo que você viu na aula 02, o que essa linha quebra?
2. O `sorted()` no vocabulário é decorativo ou necessário? O que aconteceria sem ele?
3. Olhando a matriz: qual informação do texto original foi perdida para sempre?

1-basicamente o que está sendo feito é normalizar as frases e depois disso seprar em palvras. Ou seja, o 'Não gostei' se torna um dicionario com 'não' e 'gostei', cada uma dessas palavras será um token. Essa tokenização quebra a semantica do negativa de uma frase, piis trata duas palavras indepdentes, mas elas são correlacionadas

2- esse sorted colocou em ordem alfabetica e tambem deixou apenas as palavras unicas, com isso ela foi essencial pois se nao fosse esse metodo teria varias 

3- o não

---

**Correção — slide 15**

**1. Alvo errado, conceito certo.** O que você descreveu (perder a relação entre `não` e `gostei`) é verdade sobre o **BoW**, não sobre a tokenização — e é exatamente a resposta da pergunta 3.

A linha `.lower().replace(',','').split()` só remove **vírgula**. Neste corpus ela passa limpa, porque só há vírgulas. Em qualquer texto real, não:

```
'Entrega rápida! O Sr. Silva não gostou.'
→ ['entrega', 'rápida!', 'o', 'sr.', 'silva', 'não', 'gostou.']
```

`rápida!` e `rápida` viram **colunas diferentes** no vocabulário. É o slide 30 da aula 02 de novo.

**2. Você trocou o responsável.** Quem remove as repetidas é o `set()`, não o `sorted()`.

Mas o `sorted()` também não é decorativo, por outro motivo: a ordem de iteração de um `set` **muda entre execuções** (hash randomizado por processo). Rodei o mesmo `set` em três processos:

```
['o', 'produto', 'rapida', 'adorei', 'entrega']
['produto', 'rapida', 'entrega', 'adorei', 'o']
['adorei', 'produto', 'entrega', 'o', 'rapida']
```

Sem `sorted()`, a mesma frase viraria um vetor diferente a cada execução e um vocabulário salvo em disco não serviria para vetorizar amanhã. O `sorted()` é o que torna a vetorização **reprodutível**.

**3. `não` não se perde aqui.** Se aparecesse no corpus, viraria coluna como qualquer outra palavra. Quem apaga `não` é a **remoção de stop words**, que é um passo diferente (aula 02) e não está acontecendo neste slide.

O que se perde para sempre é a **ordem das palavras**:

```
'produto bom, não quebrado'   → linha idêntica
'produto quebrado, não bom'   → linha idêntica
```

Essa era a sua resposta da pergunta 1. Você tinha o conceito, colocou na pergunta errada.


## Slide 16 — BoW com scikit-learn

> ⚠️ Esta célula **redefine `corpus`** (agora 4 documentos) e cria `X`, usado pelos slides 18, 19 e 32.

In [3]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
corpus = [
    'Adorei o produto, entrega rápida e qualidade boa.',
    'Produto chegou quebrado, qualidade péssima.',
    'Entrega rápida, atendimento bom, recomendo!',
    'Produto bom mas atendimento péssimo.',
]
# CountVectorizer: tokenização + vocabulário + matriz
vectorizer = CountVectorizer(lowercase=True, max_features=20)
X = vectorizer.fit_transform(corpus)
vocab = vectorizer.get_feature_names_out()
df = pd.DataFrame(X.toarray(), columns=vocab)
print(df)
print(f'Tamanho do vocabulário: {len(vocab)}')
print(f'Esparsidade: {1 - X.nnz / (X.shape[0] * X.shape[1]):.2%}')

   adorei  atendimento  boa  bom  chegou  entrega  mas  produto  péssima  \
0       1            0    1    0       0        1    0        1        0   
1       0            0    0    0       1        0    0        1        1   
2       0            1    0    1       0        1    0        0        0   
3       0            1    0    1       0        0    1        1        0   

   péssimo  qualidade  quebrado  recomendo  rápida  
0        0          1         0          0       1  
1        0          1         1          0       0  
2        0          0         0          1       1  
3        1          0         0          0       0  
Tamanho do vocabulário: 14
Esparsidade: 62.50%


### Sua vez

1. O slide 17 afirma que este corpus dá **vocabulário 17** e **esparsidade 73,53%**. Anote o que saiu de verdade. Por que difere?
2. O corpus tem a palavra `o` (em "Adorei o produto"). Ela está no vocabulário? Procure. Se não está, o que o `CountVectorizer` fez com ela — e por quê?
3. `max_features=20` teve algum efeito aqui? Como você comprova isso sem adivinhar?

1 - no resultado deu tamanho do vocabulario 14 e esparsidade aprox 65 
2 - não está no vocabulario, o que aconteceu foi que como é uma palavra repetida varias vezes, ela nao acrescenta 
3- não sei 

_

---

**Correção — slide 16**

**1. Vocabulário 14 ✓. Esparsidade 62,50%** (21 não-zeros em 56 células), não 65.

Faltou a segunda metade da pergunta: *por que difere do slide?* Porque **o 17 do slide não sai deste corpus de jeito nenhum**. Contando toda palavra do texto dá 16; o `CountVectorizer` entrega 14. Não existe 17 em lugar nenhum — o comentário foi escrito para outro corpus.

**2. Acertou que sumiu, errou o motivo.** Não tem nada a ver com repetição ou com a palavra "não acrescentar".

O `CountVectorizer` usa `token_pattern` padrão `(?u)\b\w\w+\b`, que exige **duas letras ou mais**. `o` e `e` têm uma letra e são descartados **antes** de qualquer contagem. É puramente o tamanho do token:

```
16 palavras no texto − 2 de uma letra ('o', 'e') = 14 no vocabulário
```

Guarde para o Tech Challenge: toda palavra de uma letra some em silêncio. Em review de e-commerce isso inclui `é`.

**3. Efeito nenhum — e dá para provar sem adivinhar.** Rode o mesmo `CountVectorizer` sem `max_features` e compare o tamanho do vocabulário:

```
sem max_features     → 14
com max_features=20  → 14
```

`max_features=20` só corta quando o vocabulário passa de 20. Com 14, não há o que cortar. A técnica que você estava procurando é essa: **mudar um parâmetro por vez e comparar**, em vez de inspecionar o resultado sozinho.


## Slide 17 — Saída típica do CountVectorizer

> Slide **sem código executável**. É um comentário descrevendo a saída esperada do slide 16.

O slide também lista os parâmetros que voltam mais adiante: `stop_words`, `max_features`, `min_df` / `max_df`.

### Sua vez

Você já usou `stop_words` na aula 01 (slide 56) e descobriu na aula 02 o que ele faz com a negação. Em uma linha: em que tipo de tarefa esse parâmetro ajuda, e em qual ele atrapalha?

onde há negação não ajuda, onde há diversas palavras comuns como 'de' ela ajuda na limpagem da base de dados, pois essas palavras nao possuem valor semantico para o modelo

_

---

**Correção — slide 17**

Certo, e a fronteira que você traçou é a certa. Dois ajustes:

**Vocabulário:** o ganho não é "limpeza da base de dados" — é reduzir **dimensionalidade e ruído** num modelo que representa frequência. A base continua a mesma; quem encolhe é a matriz.

**Precisão da regra**, que você vai usar no Tech Challenge: ajuda em **BoW/TF-IDF para classificação**; atrapalha em qualquer coisa onde ordem e negação carregam sentido — **retrieval denso incluído**. Não é "onde há negação": é onde o significado depende de palavras funcionais, e negação é só o caso mais visível.


## Slide 18 — Esparsidade

> ⚠️ Usa o `X` do slide 16. Rode o slide 16 antes.

In [4]:
# Visualizando a esparsidade
print(f'Shape: {X.shape}')                # (10000, 30000)
print(f'Total de células: {X.shape[0] * X.shape[1]:,}')  # 300,000,000
print(f'Não-zeros: {X.nnz:,}')             # ~1,500,000
print(f'Esparsidade: {1 - X.nnz/(X.shape[0]*X.shape[1]):.2%}')
# Esparsidade: 99.50%
# Armazenar matriz cheia em numpy: ~2.4 GB (300M × 8 bytes)
# Armazenar como esparsa: ~36 MB (1.5M × 24 bytes por trio)

Shape: (4, 14)
Total de células: 56
Não-zeros: 21
Esparsidade: 62.50%


### Sua vez

1. Os comentários falam em 300 milhões de células e 99,50%. O que saiu na sua tela? O que explica a diferença?
2. A conta de memória do slide (2,4 GB × 36 MB) usa 8 bytes por célula densa e 24 bytes por não-zero na esparsa. Por que o formato esparso gasta **três vezes mais** por elemento armazenado — e mesmo assim compensa?
3. A partir de qual nível de esparsidade a matriz esparsa deixa de compensar? Faça a conta com os números do slide.

1- saiu 56 ceulas com esparsidade de aprox 60, acredito que a base que estamos analisando é menor para estar mais otimizado e mais leve
2- compensa por que no final os dados ter menor tamanho, um é multiplicado por 300 outro por 1,5 ... isto pois a espasa retira os zeros 
3- quando a matriz esparsa foi apenas 3 vezes menos que a matriz cheia estamos falando do mesmmo tamanho de dados

_

---

**Correção — slide 18**

**1. Certo.** 56 células, 62,50%. A causa é essa mesma: os comentários descrevem um corpus hipotético de 10.000 × 30.000 e o seu `X` é 4 × 14. O slide comentou números de um cenário que o código não executa.

**2. Você explicou por que compensa, mas não por que são 24 bytes** — que era a pergunta.

Na matriz densa a posição é **implícita**: o valor na coluna 5 está no 5º lugar do array, então você guarda só o número (8 bytes). Na esparsa os valores estão fora de ordem e espalhados, então cada um precisa carregar **a própria coordenada**: valor + índice da coluna + a estrutura de ponteiros de linha ≈ 24 bytes.

Resumindo: você paga **3× por elemento** para poder guardar **200× menos elementos**.

**3. O mecanismo está certo, falta o número.** Empate quando o custo dos dois se iguala:

```
24 × não-zeros = 8 × total
não-zeros / total = 1/3
densidade 33,3%  →  esparsidade 66,7%
```

Abaixo de ~66,7% de esparsidade, a matriz **densa** sai mais barata.

Agora repare onde isso coloca o slide 16: 62,50% de esparsidade está **abaixo** do ponto de equilíbrio. Naquela matriz específica, o formato esparso gasta mais memória que o denso — o slide usa como exemplo de esparsidade uma matriz que não compensa armazenar como esparsa.


## Slide 19 — scipy.sparse

> 🔴 **Esta célula não roda como está no slide:** `model.fit(X, y)` usa um `y` que nenhum slide define — `NameError`.
>
> Adicionei os rótulos das 4 reviews do slide 16, marcados com `# [meu]`. Confira se você concorda com a rotulagem antes de rodar.

In [5]:
from scipy.sparse import csr_matrix
import numpy as np
# X já vem como matriz esparsa do CountVectorizer
print(type(X))  # scipy.sparse.csr_matrix
y = [1, 0, 1, 0]  # [meu] rótulos das 4 reviews do slide 16 — o slide não define y
# Usar com modelos de sklearn — todos suportam esparsa nativamente
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X, y)  # ✅ Funciona direto, sem converter
# Converter para denso APENAS se precisar visualizar
# (cuidado: pode estourar memória em corpora grandes!)
X_dense = X.toarray()  # ⚠️ Use só com X pequeno
print('tipo depois do toarray():', type(X_dense))

<class 'scipy.sparse._csr.csr_matrix'>
tipo depois do toarray(): <class 'numpy.ndarray'>


### Sua vez

1. `print(type(X))` — o slide diz `csr_matrix`. O que apareceu de fato?
2. Você vem do backend: `csr_matrix` guarda três vetores em vez da matriz inteira. Qual estrutura de dados que você já usa em C#/SQL tem a mesma ideia?
3. O modelo treinou com 4 documentos e 2 classes. O resultado significa alguma coisa? Por quê?

1 - <class 'scipy.sparse._csr.csr_matrix'>
2 - nao sei 
3 - nao sei, nao etendi muito bem esse slide e as perguntas
_

---

**Correção — slide 19**

**1. Certo.** `scipy.sparse._csr.csr_matrix` — o `_csr` é só o módulo interno; a classe é a que o slide diz.

**2. A ideia é guardar só o que existe, em vez de reservar espaço para tudo.**

Em C#: `Dictionary<int, double>` no lugar de `double[]` com quase tudo zero. Você guarda a chave junto com o valor — paga mais por item e economiza no total. É a mesma troca da pergunta 2 do slide 18.

Em SQL: uma tabela chave-valor (`entidade`, `atributo`, `valor`) no lugar de uma tabela larga com 300 colunas quase todas `NULL`.

**3. Não significa nada — e você fez bem em desconfiar.**

São 4 documentos e 14 features: **mais features que amostras**. Com isso o modelo separa perfeitamente qualquer rotulagem que você der, inclusive uma aleatória, e não generaliza nada. Não há resultado para ler ali.

E você está certo sobre o slide: ele não está mostrando um experimento. Está mostrando **uma só coisa mecânica** — que o `sklearn` aceita matriz esparsa direto no `.fit()`, sem precisar de `.toarray()`. O `model.fit` é a demonstração da compatibilidade, não um treino de verdade. As perguntas pareceram vagas porque o slide é mesmo raso.


---
# Bloco 2 · N-grams (slides 31–38)

## Slide 31 — Trigrams (n=3)

In [6]:
# Trigrams capturam expressões idiomáticas e contexto mais longo
from sklearn.feature_extraction.text import CountVectorizer
textos = [
    'não gostei nada do produto',
    'muito bom mesmo, recomendo',
    'foi de jeito nenhum',
]
# ngram_range=(3, 3) → APENAS trigrams
vec = CountVectorizer(ngram_range=(3, 3))
X = vec.fit_transform(textos)
print(vec.get_feature_names_out())
# Saída:
# ['de jeito nenhum', 'do produto'... 'mesmo recomendo',
#  'muito bom mesmo', 'não gostei nada']
# Trigrams capturam:
# - Expressões fixas: 'de jeito nenhum'
# - Modificadores intensivos: 'muito bom mesmo'
# - Negações compostas: 'não gostei nada'

['bom mesmo recomendo' 'de jeito nenhum' 'foi de jeito' 'gostei nada do'
 'muito bom mesmo' 'nada do produto' 'não gostei nada']


### Sua vez

> ⚠️ Esta célula sobrescreveu `X` e `vec`. Se voltar ao bloco 1, rode o slide 16 de novo.

1. O comentário lista `'do produto'` entre os resultados. Olhe a saída: isso é um trigram? O que aconteceu?
2. Conte os trigrams do primeiro texto à mão. Bate com o que saiu?
3. Se trigrams capturam mais contexto, por que não usar sempre `ngram_range=(1, 5)`? Responda antes do slide 33.

1 - não é um trigam, isso seria um bigram, pois é o conjunto de duas palavras
2 - nao entendi
3 - pois é muito custoso, a cada aumento de grams aumenta exponencialmente o vocabulario pois tem que fazer a combinação de 3 palavras

---

#### Correção

**1 — Correto.** `'do produto'` é bigram. Faltou a segunda metade da pergunta ("o que
aconteceu?"): quem montou o slide colou saída de bigrams dentro do exemplo de trigrams.
Com `ngram_range=(3, 3)` — mínimo 3, máximo 3 — um bigram é impossível de aparecer.

**2 — É aritmética de janela deslizante.** Texto com 5 palavras, janela de 3:

```
não gostei nada do produto
└──────┬──────┘              → "não gostei nada"
    └──────┬──────┘          → "gostei nada do"
        └──────┬──────┘      → "nada do produto"
```

A janela anda uma palavra por vez: **3 trigrams**. Fórmula: `palavras − n + 1`.
O slide lista só 1 dos 3 — não bate. A saída real dos três textos tem 7 trigrams
(`'bom mesmo recomendo'`, `'de jeito nenhum'`, `'foi de jeito'`, `'gostei nada do'`,
`'muito bom mesmo'`, `'nada do produto'`, `'não gostei nada'`).

**3 — Conclusão certa, mecanismo errado em duas coisas.**

- ❌ *"combinação de 3 palavras"* → não há combinação. São **janelas contíguas na ordem do
  texto**. Combinação de 5 palavras tomadas 3 a 3 daria 10; a janela dá 3.
- ❌ *"aumenta exponencialmente"* → cada `n` soma no máximo `palavras − n + 1` por
  documento. Cresce **linearmente**, e no corpus real até desacelera.

Medido em 5.000 reviews do corpus Olist do Tech Challenge:

```
  n   features   só em 1 doc   % inútil
  1      5.596         3.125      55.8%
  2     28.498        22.626      79.4%
  3     42.353        38.881      91.8%
  4     44.374        42.934      96.8%
```

O vocabulário desacelera (de n=3 para n=4 cresce só 2 mil). O que explode é a coluna da
direita. **O problema real não é tamanho, é inutilidade:** uma feature que aparece em um só
documento nunca coincide com outro, logo não compara nada. Em `n=4` você paga memória por
44 mil colunas das quais 43 mil são peso morto.

**Resposta 3, limpa:**

> Porque n-grams são janelas contíguas (`palavras − n + 1` por documento), não combinações
> — então o vocabulário cresce e até desacelera, mas quanto maior o `n`, mais raro cada
> n-gram fica. Com `n=4` no corpus Olist, 96,8% aparecem em um só documento, e feature que
> não se repete não serve para comparar documentos. Paga-se memória por colunas que nunca
> coincidem.


## Slide 32 — range (1, 2): unigrams + bigrams

> ⚠️ Usa o `corpus` do **slide 16** (as 4 reviews). Rode o slide 16 antes.

In [7]:
# CountVectorizer com unigrams + bigrams
vec = CountVectorizer(ngram_range=(1, 2), min_df=2)
X = vec.fit_transform(corpus)
# Vocabulário agora inclui AMBOS:
#   ['adorei', 'adorei produto', 'bom', 'entrega',
#    'entrega rápida', 'gostei', 'não', 'não gostei',
#    'produto', 'rápida', ...]
# Em sentiment analysis em PT-BR:
# - Unigrams capturam 'gostei', 'péssimo', 'maravilhoso'
# - Bigrams capturam 'não gostei', 'muito bom', 'super ruim'
print(f'Total de features: {len(vec.get_feature_names_out())}')
print(vec.get_feature_names_out())

Total de features: 7
['atendimento' 'bom' 'entrega' 'entrega rápida' 'produto' 'qualidade'
 'rápida']


### Sua vez

1. O comentário promete `'não gostei'` e `'adorei produto'` no vocabulário. Eles estão lá? Procure o motivo em **duas** causas diferentes — uma é o corpus, a outra é um parâmetro.
2. `min_df=2` num corpus de 4 documentos: o que sobrou e por quê?
3. Se o objetivo é capturar `'não gostei'`, que mudança mínima você faria nesta célula?

1 - o 'ngram_range=(1, 2)' determina que o gram seja minimo 1 e maximo 2. Mas os exemplos dados nao aparecem no resultado, isso porque o corpus que estamos usando é menor do que o esperado

2 - captura no minimo 2 documentos, com isso pode ter capturado 2 e nao apareceu o que queriamos 

3 - colocaria no min 4 dfs para abranger todos

---

#### Correção

Mesmo padrão do slide 31: você descreve um efeito real e atribui à peça errada. É o ponto
a treinar.

**1 — As duas causas são independentes, e você juntou tudo em "corpus menor".**

O enunciado pedia duas causas porque **cada termo prometido falta por um motivo diferente**:

- **`'não gostei'`** → causa é o **corpus**: a palavra `não` **não existe em nenhum dos 4
  documentos**. Mesmo com `min_df=1` nenhum bigram com `não` aparece. Não é "corpus menor
  que o esperado" — é corpus que não contém a palavra. O comentário do slide descreve um
  corpus que não é este.
- **`'adorei produto'`** → causa é o **parâmetro**: o bigram **existe** (com `min_df=1` ele
  está lá), mas `adorei` aparece em **1 só documento**, e `min_df=2` o elimina.

Note também: você começou explicando o `ngram_range=(1, 2)`. Ele não tem culpa nenhuma aqui
— está correto e gerando os bigrams. Apontar o componente inocente é exatamente o erro de
atribuição.

**2 — Primeira metade certa, segunda confusa.** `min_df=2` = "manter só o que aparece em ao
menos 2 documentos". Correto. O que sobrou:

```
7 features: ['atendimento', 'bom', 'entrega', 'entrega rápida', 'produto', 'qualidade', 'rápida']
                                              └─ o ÚNICO bigram sobrevivente
```

O dado que importa: de 30 features com `min_df=1`, sobraram 7 — e **só 1 bigram**. Bigram
precisa que *duas palavras na mesma ordem* se repitam em outro documento, o que é muito mais
raro que uma palavra isolada se repetir. É a mesma lição do slide 31 aparecendo de novo:
quanto maior o `n`, mais raro o n-gram.

**3 — Invertido.** `min_df=4` é **mais** restritivo, não menos: exigiria o termo aparecer em
**todos** os 4 documentos. O resultado real:

```
ValueError: After pruning, no terms remain. Try a lower min_df or a higher max_df.
```

Zera o vocabulário e quebra a célula. `min_df` é um **piso de raridade** — subir o piso corta
mais, não abrange mais.

E a pegadinha honesta da pergunta: `min_df=1` é **necessário mas não suficiente**. Com o
corpus do slide 16 não existe `não` para capturar. A mudança mínima real é **trocar o
corpus** por um que contenha a expressão — é o que o slide 33 faz, e lá `'não gostei'` é
capturado com `min_df=1`.

**Respostas limpas:**

> **1.** Não. `'não gostei'` falta porque a palavra `não` não existe no corpus — nem com
> `min_df=1` ela aparece. `'adorei produto'` existe como bigram, mas `adorei` está em só 1
> documento e `min_df=2` o elimina. Uma causa é o dado, a outra é o parâmetro.
>
> **2.** `min_df=2` mantém só o que aparece em ao menos 2 documentos: 7 features de 30, das
> quais **um único bigram** (`'entrega rápida'`). Bigram exige que duas palavras na mesma
> ordem se repitam entre documentos — bem mais raro que uma palavra isolada.
>
> **3.** `min_df=1`, para não podar o que aparece uma vez só. Mas não basta: o corpus do
> slide 16 não tem `não`. Para capturar de fato, é preciso um corpus que contenha a
> expressão.


## Slide 33 — Implementação prática

> ⚠️ **Redefine `corpus`** de novo (4 novos documentos).

In [8]:
from sklearn.feature_extraction.text import CountVectorizer
corpus = [
    'Adorei o produto, super recomendo',
    'Não gostei muito do produto',
    'Produto bom mesmo, vale o preço',
    'Muito ruim, não comprem',
]
# Comparando ngram_range diferentes
for n_range in [(1, 1), (1, 2), (1, 3), (2, 2)]:
    vec = CountVectorizer(ngram_range=n_range)
    X = vec.fit_transform(corpus)
    print(f'ngram_range={n_range}: {X.shape[1]} features')
# Saída típica:
# ngram_range=(1, 1): 18 features    ← unigrams só
# ngram_range=(1, 2): 35 features    ← +17 bigrams
# ngram_range=(1, 3): 50 features    ← +15 trigrams
# ngram_range=(2, 2): 17 features    ← APENAS bigrams

ngram_range=(1, 1): 14 features
ngram_range=(1, 2): 28 features
ngram_range=(1, 3): 38 features
ngram_range=(2, 2): 14 features


### Sua vez

1. Anote os quatro números reais ao lado dos quatro do comentário. Todos divergem — mas a **relação entre eles** se mantém? Qual relação?
2. Este é o dado que responde a pergunta 3 do slide 31. Com ele, formule a regra: o que acontece com o vocabulário conforme `n` cresce, e por que isso é um problema?
3. `(1,2)` tem 28 features e `(2,2)` tem 14. Some 14 de `(1,1)` e veja. O que essa aritmética te diz sobre como o `ngram_range` funciona?

1 - Sim, a relação se mantem, quanto se aumenta o n maximo vai aumentando as features e se o n maximo e minimo forem iguais a quantidade de features é a mesma 

2 - se for aumentando o n vai aumentando as features e cada vez mais dificil encontrar similaridade, por exemplo, n=3 sendo 'nao gostei do' vai ser mais dificil de ser encontrado em outros documentos do que 'nao gostei'


3 - me diz que quanto a maior  a diferenca entre o min e max maior a quantidade de features e se nao houver diferença, por exemplo (1,1) ou (2,2) as features são de valor igual

---

#### Correção

**1 — Faltou a primeira metade, e a regra que você formulou é falsa.**

A pergunta pedia os quatro números anotados. São estes:

| `ngram_range` | slide diz | real |
|---|---|---|
| `(1,1)` | 18 | **14** |
| `(1,2)` | 35 | **28** |
| `(1,3)` | 50 | **38** |
| `(2,2)` | 17 | **14** |

A relação que se mantém é **aditividade**:

```
real :  14 + 14 = 28   → (1,1) + (2,2) = (1,2)
        28 + 10 = 38   → (1,2) + (3,3) = (1,3)
slide:  18 + 17 = 35   ✓
        35 + 15 = 50   ✓
```

Os números absolutos do slide estão errados — vieram de outro corpus — mas a **aritmética
interna dele é coerente**. Era isso que "a relação se mantém" queria dizer.

Já a sua regra — *"se o n mínimo e máximo forem iguais, a quantidade de features é a mesma"* —
nasceu de uma coincidência. `(1,1)` e `(2,2)` dão 14 por acaso **neste** corpus. Basta olhar
um passo além:

```
(3,3) → 10 features
(4,4) →  6 features
```

Mesma condição (min = max), contagens diferentes. O contraexemplo estava a um parâmetro de
distância.

**2 — A segunda metade está certa, e é o ponto principal.**

*"Cada vez mais difícil encontrar similaridade"* é exatamente a razão, e o exemplo
`'não gostei do'` contra `'não gostei'` é bom — é o mecanismo correto.

A primeira metade repete o erro que você já levou no slide 31. São **dois contadores
diferentes**:

```
faixa cumulativa (1,n):  14 → 28 → 38 → 44    cresce
n isolado        (n,n):  14 → 14 → 10 →  6    ENCOLHE
```

Os documentos deste corpus têm de 4 a 6 palavras; uma janela de 4 quase não cabe. O
vocabulário de cada `n` **diminui** enquanto a inutilidade aumenta. O problema nunca foi
tamanho — é raridade, que é justamente o que você disse na segunda metade.

**3 — Você passou ao lado da aritmética.** A conta pedida era literal:

```
14 (unigramas) + 14 (bigramas)  = 28 = (1,2)
28             + 10 (trigramas) = 38 = (1,3)
```

O que ela revela: `ngram_range=(a, b)` **não cria um tipo novo de feature** — ele empilha os
conjuntos de `n = a` até `n = b`. É união, não interação. Por isso `(1,2)` é exatamente
unigramas mais bigramas, sem sobra e sem sobreposição: um unigrama nunca pode ser igual a um
bigrama.

Sua resposta (*"quanto maior a diferença entre min e max, maior a quantidade"*) é verdadeira
como tendência, mas é **consequência**, não mecanismo — e a segunda metade repete a regra
falsa da pergunta 1.

**Respostas limpas:**

> **1.** `(1,1)=14`, `(1,2)=28`, `(1,3)=38`, `(2,2)=14` — todos divergem do slide. A relação
> que se mantém é a aditividade: `(1,1)+(2,2)=(1,2)` e `(1,2)+(3,3)=(1,3)`, tanto nos números
> reais quanto nos do slide.
>
> **2.** A faixa cumulativa cresce, mas o vocabulário de cada `n` isolado encolhe em
> documentos curtos. O problema é **raridade**: quanto maior o `n`, menor a chance de o
> n-grama se repetir em outro documento — e feature que não se repete não compara nada.
>
> **3.** `ngram_range=(a,b)` é a união dos conjuntos de n-gramas de `a` até `b`. Não cria
> features novas, empilha as que já existem.


## Slide 35 — Filtros min_df e max_df

> 🔴 **Não roda como está no slide.** O `corpus` vigente tem 4 documentos e o slide pede `min_df=5` — nenhum termo pode aparecer em 5 documentos de 4. Estoura `ValueError: max_df corresponds to < documents than min_df`.
>
> Apontei para o `reviews_corpus` (64 docs), marcado com `# [meu]`. O resto é do slide.

In [20]:
# Filtros para podar n-grams ruidosos ou inúteis
vec = CountVectorizer(
    ngram_range=(1, 2),
    min_df=5,        # ignora termos em MENOS de 5 documentos
    max_df=0.5,     # ignora termos em MAIS de 85% dos documentos
    max_features=10000,  # limita vocabulário aos 10k mais frequentes
)
X = vec.fit_transform(reviews_corpus)  # [meu] o slide usa `corpus` (4 docs) e estoura
print('features:', X.shape[1])
print(vec.get_feature_names_out())
# Por que filtrar?
# - min_df remove n-grams raros (typos, palavras únicas) → reduz ruído
# - max_df remove palavras universais (alternativa a stop words)
# - max_features cap absoluto no tamanho do vocabulário
# Exemplo: corpus de 10k docs
# Sem filtros: ~600k features (lento!)
# Com filtros acima: ~10k features (manejável)

features: 39
['atendimento' 'boa' 'bom' 'com' 'da' 'demorou' 'demorou muito' 'do'
 'entrega' 'entrega rápida' 'gostei' 'gostei da' 'gostei do'
 'gostei muito' 'mas' 'muito' 'muito não' 'não demorou' 'não gostei'
 'não recomendo' 'não tive' 'não veio' 'problema' 'problema com' 'produto'
 'produto bom' 'qualidade' 'qualidade boa' 'qualidade ruim' 'quebrado'
 'recomendo' 'recomendo muito' 'ruim' 'rápida' 'tive' 'tive problema'
 'veio' 'veio quebrado' 'vendedor']


### Sua vez

1. Rode de novo com `min_df=2` e compare o vocabulário. O que `min_df=5` jogou fora?
2. O slide chama `max_df` de "alternativa a stop words". Compare as duas abordagens: qual a vantagem de `max_df` sobre uma lista fixa como a do NLTK? E a desvantagem?
3. Rode com `max_df=0.5`. A palavra `não` sobrevive? Ela está em quantos por cento dos documentos do `reviews_corpus`? Confira.

1 - o min_df=5 jogou fora features que não apareecem em pelo menos 5 documentos

2 - se tiver um termo que está presente em mais de 85% dos documentos ele é descartado. Isso acontece porque a probabilidade desse termo nao acrescentar semanticamente no modelo é alta. Em comparação com o NLTK já temos termos fixos que são descartados como 'e', 'de' e etc. E a diferença com o max_df é que se 'produto' aparecer muitas vezes essa palavra é descartada e no NLTK não, mesmo sabendo que essa palavra nao acrescenta no modelo por aparecer tantas vezes.Vejo que a desvantagem é que talvez termos como 'de' fiquem presentes para serem processados pois estao abaixo da porcentagem de 85% e nao possuem valor semantico significativo

3 - a palavra 'não' não sobrevive, apenas em bigrams com 'não tive', por exemplo. Que insight tiro com isso?

---

#### Correção

**1 — Você repetiu a definição do parâmetro em vez de observar o efeito.** A pergunta era
*o que* ele jogou fora. Rodando os três:

```
min_df=1  → 167 features
min_df=2  →  85
min_df=5  →  39
```

Do corte de 2 para 5 saíram 46 termos, e a composição é o ponto:

```
37 bigramas  (80%)
 9 unigramas (20%)
```

Sobraram 21 unigramas e 18 bigramas. O piso de frequência **mata bigrama muito antes de
matar unigrama** — pelo motivo que você mesmo formulou no slide 33: bigrama exige que *duas
palavras na mesma ordem* se repitam, o que é bem mais raro. É a terceira vez que essa lição
reaparece na aula.

**2 — Certo nas duas metades, e é a sua melhor resposta até agora.**

A vantagem que você apontou é a principal: `max_df` é **relativo ao corpus**, a lista do NLTK
é fixa. Num corpus de reviews de produto, `produto` vira ruído e nenhuma stop list genérica
saberia disso.

A desvantagem que você levantou **não é hipotética — é mensurável aqui**. Com `max_df=0.85`,
o filtro cortou exatamente **1 termo de 167**:

```
não       → 64/64 docs (100%)   ← único cortado
muito     → 48%
gostei    → 47%
recomendo → 41%
produto   → 38%
```

Todo o resto está longe do teto. `max_df=0.85` quase não fez nada neste corpus, exatamente
como você previu.

Uma peça que falta na sua formulação: `max_df` não mede "vazio de semântica", mede
**ubiquidade** — são coisas diferentes. Termo raro e inútil (typo, número de pedido) passa
ileso pelo `max_df`; quem pega esse é o `min_df`. Os dois filtros não são versões do mesmo
filtro, são filtros de pontas opostas.

**3 — Sua observação está certa, e sua desconfiança também. São duas coisas separadas.**

**O insight real, e transferível:** o filtro é aplicado a **cada n-grama de forma
independente**. `não` sozinho tem DF de 100% e é cortado. `não tive` é um termo *diferente*,
com DF próprio bem menor, e por isso sobrevive. **Uma palavra eliminada como unigrama
continua viva dentro dos bigramas que a contêm.**

Isso tem consequência direta no que você viu na aula 02: remover `não` destrói a negação —
mas só em unigramas. Com `ngram_range=(1, 2)`, `não gostei` e `não recomendo` atravessam o
corte e a negação continua representada. Guarde para o slide 38.

**E a parte que justifica a sua pergunta "que insight tiro com isso?":** nenhum, quanto ao
percentual. O `não` estar em **100%** dos documentos não é fato sobre NLP, é artefato meu —
construí as 64 reviews todas com negação. Num corpus real não acontece: no Olist, 25,9% das
reviews têm marcador de negação. Perguntar "em quantos por cento está?" só te fazia investigar
a minha fixture. Era uma das perguntas que você mandou tirar, e a reescrita se perdeu antes de
chegar até você. Seu instinto estava certo.

**Respostas limpas:**

> **1.** De `min_df=2` para `min_df=5` o vocabulário cai de 85 para 39 features. Dos 46 termos
> cortados, 37 são bigramas e 9 unigramas — o piso de frequência elimina bigramas primeiro,
> porque repetir duas palavras na mesma ordem entre documentos é mais raro que repetir uma.
>
> **2.** `max_df` é relativo ao corpus e pega o que é ubíquo *naquele* domínio (`produto` num
> corpus de reviews), coisa que uma lista fixa nunca saberia. A desvantagem é que ele só vê
> ubiquidade: palavra funcional abaixo do teto sobrevive — neste corpus `max_df=0.85` cortou
> um único termo de 167 — e termo raro e inútil não é problema dele, e sim do `min_df`.
>
> **3.** Não sobrevive como unigrama, mas sobrevive dentro dos bigramas. O filtro avalia cada
> n-grama separadamente, com o DF daquele n-grama — então eliminar uma palavra sozinha não a
> elimina das combinações em que ela aparece.


## Slide 38 — Bigrams + sentiment analysis

> 🔴 **Não roda como está no slide:** `reviews_corpus` e `labels` estão comentados no original. Usa o corpus do topo do notebook.

Antes de rodar: eu disse lá em cima que as palavras isoladas estão balanceadas entre as duas classes. Escreva agora sua previsão do ganho dos bigrams — maior, menor ou igual aos 2–5 pontos que o slide promete?

In [21]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
# Corpus de reviews + labels (0=neg, 1=pos)
# reviews_corpus = [...]; labels = [...]     # [meu] definidos no topo do notebook
# Sem bigrams (unigrams puros)
bow_uni = CountVectorizer(ngram_range=(1, 1), min_df=5)
X_uni = bow_uni.fit_transform(reviews_corpus)
f1_uni = cross_val_score(LogisticRegression(max_iter=1000),
                          X_uni, labels, cv=5, scoring='f1').mean()
# Com unigrams + bigrams
bow_bi = CountVectorizer(ngram_range=(1, 2), min_df=5)
X_bi = bow_bi.fit_transform(reviews_corpus)
f1_bi = cross_val_score(LogisticRegression(max_iter=1000),
                         X_bi, labels, cv=5, scoring='f1').mean()
print(f'Unigrams puros:    F1 = {f1_uni:.3f}')
print(f'Unigrams + bigrams: F1 = {f1_bi:.3f}')
print(f'Ganho: +{(f1_bi - f1_uni)*100:.1f} pontos')
# Tipicamente: ganho de 2-5 pontos em sentiment PT-BR

Unigrams puros:    F1 = 0.774
Unigrams + bigrams: F1 = 0.889
Ganho: +11.5 pontos


### Sua vez

1. Sua previsão bateu? O ganho foi bem maior que os 2–5 pontos do slide. **Por que este corpus exagera o efeito?** (a resposta está em como eu o construí)
2. Rode a célula abaixo e olhe quais bigrams sobreviveram ao `min_df=5`. Quais deles um modelo de unigramas **não** conseguiria representar?
3. Pergunta que fecha o bloco: se bigramas resolvem negação, por que a aula 02 tratou a negação como problema sem solução?

_

_

In [11]:
# [meu] apoio para a pergunta 2 — não faz parte do slide
print('unigramas:', bow_uni.get_feature_names_out())
print()
bigramas = [f for f in bow_bi.get_feature_names_out() if ' ' in f]
print(f'bigramas que sobreviveram ({len(bigramas)}):', bigramas)

unigramas: ['atendimento' 'boa' 'bom' 'com' 'da' 'demorou' 'do' 'entrega' 'gostei'
 'mas' 'muito' 'não' 'problema' 'produto' 'qualidade' 'quebrado'
 'recomendo' 'ruim' 'rápida' 'tive' 'veio' 'vendedor']

bigramas que sobreviveram (18): ['demorou muito', 'entrega rápida', 'gostei da', 'gostei do', 'gostei muito', 'muito não', 'não demorou', 'não gostei', 'não recomendo', 'não tive', 'não veio', 'problema com', 'produto bom', 'qualidade boa', 'qualidade ruim', 'recomendo muito', 'tive problema', 'veio quebrado']


---
# Bloco 3 · TF-IDF (slides 48–53)

## Slide 48 — TF: Term Frequency

In [12]:
def tf(palavra, doc):
    """Term Frequency normalizado pelo tamanho do doc."""
    palavras = doc.split()
    return palavras.count(palavra) / len(palavras)
doc = 'o produto bom o produto rápido'
# 6 palavras totais
print(tf('o', doc))         # 2/6 = 0.333
print(tf('produto', doc))   # 2/6 = 0.333
print(tf('bom', doc))       # 1/6 = 0.167
print(tf('rápido', doc))    # 1/6 = 0.167

0.3333333333333333
0.3333333333333333
0.16666666666666666
0.16666666666666666


### Sua vez

1. Por que dividir pelo tamanho do documento? O que quebraria num corpus de reviews se usássemos a contagem crua?
2. `o` e `produto` empatam em 0.333. Qual das duas diz mais sobre o documento? O TF sozinho consegue distinguir?
3. A pergunta 2 é exatamente o buraco que o próximo slide preenche. Antes de vê-lo: que informação **de fora deste documento** você precisaria para resolver?

_

_

## Slide 49 — IDF: Inverse Document Frequency

In [13]:
import numpy as np
def idf(palavra, corpus):
    """Inverse Document Frequency."""
    df = sum(1 for doc in corpus if palavra in doc.split())
    return np.log(len(corpus) / df) if df > 0 else 0
corpus = [
    'o produto bom',
    'o produto ruim',
    'o produto quebrado',
    'o produto rápido',
]
print(idf('o', corpus))         # log(4/4) = 0     ← em TODO doc
print(idf('produto', corpus))   # log(4/4) = 0     ← em TODO doc
print(idf('bom', corpus))       # log(4/1) = 1.39  ← só em 1 doc

0.0
0.0
1.3862943611198906


### Sua vez

1. IDF de `o` deu exatamente 0. Multiplicado pelo TF, o peso final é 0. Descreva o que isso faz com as stop words — e compare com a abordagem da aula 02.
2. Por que `log`? Teste: troque por `len(corpus)/df` puro e compare os pesos de `bom` num corpus de 4 e num de 10.000 documentos.
3. `idf` retorna 0 quando `df == 0`. Quando isso acontece na prática, e por que 0 é uma escolha perigosa aqui?

_

_

## Slide 51 — TF-IDF manualmente

> ⚠️ **Redefine `corpus`** de novo.

In [14]:
import numpy as np
import pandas as pd
corpus = [
    'produto bom rapida entrega',
    'produto ruim quebrado',
    'entrega rapida atendimento bom',
    'atendimento ruim produto bom',
]
vocab = sorted(set(p for doc in corpus for p in doc.split()))
def tf(p, doc):
    palavras = doc.split()
    return palavras.count(p) / len(palavras)
def idf(p, corpus):
    df = sum(1 for doc in corpus if p in doc.split())
    return np.log(len(corpus) / df) if df > 0 else 0
# Calcular matriz TF-IDF
matriz = []
for doc in corpus:
    linha = [tf(p, doc) * idf(p, corpus) for p in vocab]
    matriz.append(linha)
df_tfidf = pd.DataFrame(matriz, columns=vocab)
print(df_tfidf.round(3))

   atendimento    bom  entrega  produto  quebrado  rapida   ruim
0        0.000  0.072    0.173    0.072     0.000   0.173  0.000
1        0.000  0.000    0.000    0.096     0.462   0.000  0.231
2        0.173  0.072    0.173    0.000     0.000   0.173  0.000
3        0.173  0.072    0.000    0.072     0.000   0.000  0.173


### Sua vez

O slide 52 afirma que o maior peso da matriz é `quebrado` no Doc 2, com **0.462**.

1. Confirme na sua saída. Bateu?
2. Refaça a conta à mão para `quebrado` no Doc 2: qual o TF, qual o IDF, qual o produto?
3. A coluna `produto` aparece em 3 dos 4 documentos. Qual o peso dela, e por quê?
4. Olhe a linha do Doc 1. Há empate entre vários termos. O que isso revela sobre o poder de discriminação do TF-IDF em corpus pequeno?

_

_

## Slide 52 — Saída do TF-IDF manual

> Slide **sem código executável** — é o comentário com a saída esperada do slide 51, já usado nas perguntas acima.

A leitura que o slide propõe: `produto` está em 3/4 docs → IDF baixo → peso baixo; `quebrado` está em 1/4 → IDF alto → peso 0.462, o maior da matriz; `atendimento` em 2/4 → IDF médio.

### Sua vez

Em uma frase, com suas palavras: o que o TF-IDF mede que o BoW não mede?

_

_

## Slide 53 — TfidfVectorizer em produção

> 🔴 **Não roda como está no slide:** usa `reviews_corpus`, que o slide nunca define. Usa o corpus do topo.

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),         # unigrams + bigrams
    min_df=5,                   # ignora termos em <5 docs
    max_df=0.85,                # ignora termos em >85% dos docs
    max_features=10000,         # cap no vocabulário
    sublinear_tf=True,          # aplica log(1+TF) em vez de TF puro
)
X = vectorizer.fit_transform(reviews_corpus)  # 10k documentos
print(X.shape)        # (10000, 10000)
print(X.nnz)          # número de elementos não-zero
# Top features por documento
import numpy as np
feature_names = vectorizer.get_feature_names_out()
doc0 = X[0].toarray().flatten()
top_idx = np.argsort(doc0)[-5:][::-1]
print('Top 5 features do doc 0:')
for i in top_idx:
    print(f'  {feature_names[i]}: {doc0[i]:.3f}')

(64, 39)
522
Top 5 features do doc 0:
  rápida: 0.415
  entrega rápida: 0.415
  produto bom: 0.330
  bom: 0.320
  entrega: 0.320


### Sua vez

1. O doc 0 é `'Não tive problema, produto bom e entrega rápida'`. As 5 features de maior peso fazem sentido como resumo dele? O que ficou de fora que você esperava ver?
2. `max_df=0.85` com `não` presente em 100% dos documentos deste corpus: o que aconteceu com essa palavra? Procure na lista de features.
3. `sublinear_tf=True` troca TF por `log(1+TF)`. Em reviews de e-commerce, com textos curtos, isso muda muita coisa? Teste com `False` e compare os pesos.

_

_

---
# Bloco 4 · BoW vs TF-IDF (slide 55)

## Slide 55 — A tarefa real

> 🔴 **Não roda como está no slide:** `reviews_corpus` e `labels` não são definidos. Usa o corpus do topo.

Escreva sua previsão antes de rodar: o slide diz que TF-IDF "geralmente vence". Quanto você aposta de diferença?

In [16]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
# Hipotético: reviews_corpus e labels (0=neg, 1=pos) já preparados
# 1) BoW
bow = CountVectorizer(ngram_range=(1, 2), min_df=5)
X_bow = bow.fit_transform(reviews_corpus)
score_bow = cross_val_score(
    LogisticRegression(max_iter=1000), X_bow, labels,
    cv=5, scoring='f1'
).mean()
# 2) TF-IDF
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(reviews_corpus)
score_tfidf = cross_val_score(
    LogisticRegression(max_iter=1000), X_tfidf, labels,
    cv=5, scoring='f1'
).mean()
print(f'BoW    F1: {score_bow:.3f}')   # ex: 0.812
print(f'TF-IDF F1: {score_tfidf:.3f}') # ex: 0.834 — geralmente vence

BoW    F1: 0.889
TF-IDF F1: 0.885


### Sua vez — esta é a pergunta mais importante do notebook

**TF-IDF não venceu.** Os dois empatam, e o TF-IDF pode até ficar uma fração abaixo. Isso contradiz o slide.

Tentei construir um corpus onde o TF-IDF vencesse e não consegui — desisti de propósito, porque a explicação vale mais que o número bonito.

1. O que o IDF precisa encontrar num corpus para fazer diferença? (dica: reveja o slide 49 — o que aconteceu com `o` e `produto` lá)
2. Este corpus tem 64 documentos curtos, quase todos com o mesmo vocabulário pequeno. Isso ajuda ou atrapalha o IDF?
3. Conclua: em que condições o slide está certo, e em que condições ele está errado? Escreva a regra.
4. Você tem 33.812 reviews reais do Olist preparadas no repo do Tech Challenge. Essa é a única forma honesta de testar a afirmação do slide. Vale rodar?

_

_

---
## Fechamento — responda sem olhar para cima

1. Em uma frase cada: o que BoW representa, o que TF-IDF acrescenta, o que n-grams acrescentam.
2. Você tem um corpus de 50 mil reviews. Escolha `ngram_range`, `min_df`, `max_df` e justifique cada um.
3. A aula 02 removeu stop words por lista fixa. A aula 03 tem **dois** mecanismos que fazem um trabalho parecido sem lista. Quais são, e qual você prefere?
4. Nada nesta aula resolve a polissemia do slide 30 da aula 01 (`banco` financeiro × móvel × tech). Por quê? O que precisaria ser diferente?

_

_

## Ponte para o Tech Challenge

5. Tudo desta aula produz vetores **esparsos** de contagem. O retrieval do seu RAG vai usar vetores **densos**. Antes de estudar embeddings, escreva sua hipótese: qual a diferença fundamental entre os dois, e por que o TF-IDF não serviria para o índice?
6. Você decidiu ter dois pipelines de normalização no Tech Challenge. Depois desta aula, qual dos dois alimenta um TF-IDF — e para que você usaria um TF-IDF no projeto, já que ele não vai ser o índice?

_

_